# Manderson & Considine (2025)

Illinois county-level LOCA2 monthly indicators. Keep this notebook beside `loca2_common.py` and run from top to bottom.

Run discovery requires tasmin, tasmax, and precipitation only. Humidity is optional: missing, unreadable, or invalid humidity inputs are reported and represented by NaN; they never exclude temperature and precipitation calculations. Existing output files are always skipped and never overwritten.

Paths, scenarios, periods, and cosine-latitude county weights come from `loca2_common.py`. No population, crop, or LandScan weights are used. Missing dates in humidity are retained as NaN, without shortening the temperature series. A grid-cell month needs all humidity days to produce a humidity mean.

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [ ]:
from pathlib import Path
import gc
import os
import tempfile
import numpy as np
import xarray as xr
from tqdm.auto import tqdm
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties, daily_mean_temperature,
    monthly_mean, monthly_sum, monthly_temperature_bin_counts,
    to_fahrenheit,
)

# Filter out monthly/yearly summaries from all shared daily loaders.
if not hasattr(common, '_mc_original_nc_files'):
    common._mc_original_nc_files = common.nc_files

def daily_nc_files(directory):
    excluded = ('.monthly.', '.yearly.', '.annual.', '.seasonal.', '.us_county_avg.')
    return [p for p in common._mc_original_nc_files(directory)
            if not any(token in Path(p).name.lower() for token in excluded)]

common.nc_files = daily_nc_files

## 2. Paper settings

In [ ]:
PAPER_NAME = 'Manderson & Considine (2025)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr')
OPTIONAL_VARIABLES = ('hursmin', 'hursmax')
OVERWRITE = False
counties = common.load_counties()

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [ ]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
output_directory.mkdir(parents=True, exist_ok=True)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)

# Recognize existing runs by metadata as well as the standard filename.
existing_runs = set()
for path in output_directory.glob('*.nc'):
    with xr.open_dataset(path, decode_times=False, decode_timedelta=False) as ds:
        key = tuple(str(ds.attrs.get(k, '')) for k in ('model', 'member', 'scenario'))
        if all(key):
            existing_runs.add(key)

def output_path(model, member, scenario):
    return output_directory / (
        f'{common.safe_token(PAPER_NAME)}_'
        f'{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc'
    )

def already_exists(model, member, scenario):
    return ((model, member, scenario) in existing_runs
            or output_path(model, member, scenario).exists())

print('Input root:', common.PAPER_INPUT_ROOT)
print('Output folder:', output_directory)
print('Eligible runs without humidity requirement:', len(jobs))
print('Existing runs to skip:', sum(already_exists(*job) for job in jobs))
print('New runs to process:', sum(not already_exists(*job) for job in jobs))
if not jobs:
    raise FileNotFoundError('No runs with tasmin, tasmax and precipitation found.')
print('Counties:', len(counties))

## 4. Paper indicator calculation

In [ ]:
def humidity_on_temperature_dates(data, template):
    units = str(data.attrs.get('units', '')).strip().lower()
    if units not in {'%', 'percent', 'percentage'}:
        raise ValueError(f'{data.name}: expected percent units, found {units!r}')
    # Ignore clock-hour differences, but preserve the source calendar dates.
    dates = data.time.dt.strftime('%Y-%m-%d').values
    if len(np.unique(dates)) != len(dates):
        raise ValueError(f'{data.name}: duplicate humidity calendar dates')
    for dim in ('lat', 'lon'):
        np.testing.assert_array_equal(data[dim].values, template[dim].values)
    wanted = template.time.dt.strftime('%Y-%m-%d').values
    aligned = data.assign_coords(time=dates).reindex(time=wanted)
    return aligned.assign_coords(time=template.time)


def calculate_manderson_considine(inputs):
    """
    Calculate monthly Manderson and Considine indicators and
    aggregate the results to Illinois counties.
    """
    tmin, tmax = xr.align(inputs['tasmin'], inputs['tasmax'], join='exact')
    tmean_c = daily_mean_temperature(tmin, tmax)
    tmean_f = to_fahrenheit(tmean_c)
    edges_f = [-np.inf, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0, 85.0, np.inf]
    labels = ['tmean_days_lt15F', 'tmean_days_15_25F', 'tmean_days_25_35F', 'tmean_days_35_45F', 'tmean_days_45_55F', 'tmean_days_55_65F', 'tmean_days_65_75F', 'tmean_days_75_85F', 'tmean_days_ge85F']
    variables = monthly_temperature_bin_counts(temperature=tmean_f, edges=edges_f, labels=labels)
    variables['monthly_precipitation'] = monthly_sum(inputs['pr'])
    # Start with an all-NaN field on the temperature grid and monthly calendar.
    rh_monthly = xr.full_like(variables[labels[0]], np.nan, dtype='float64')
    humidity_status = inputs.get('_humidity_status', 'not available')
    if 'hursmin' in inputs and 'hursmax' in inputs:
        try:
            rh_min = humidity_on_temperature_dates(inputs['hursmin'], tmean_c)
            rh_max = humidity_on_temperature_dates(inputs['hursmax'], tmean_c)
            valid = (np.isfinite(rh_min) & np.isfinite(rh_max)
                     & (rh_min >= 0) & (rh_max <= 100) & (rh_min <= rh_max))
            daily_rh = (0.5 * (rh_min + rh_max)).where(valid)
            # Explicit skipna=False: missing humidity days invalidate that month.
            rh_parts = []
            for month in range(1, 13):
                selected = daily_rh.where(daily_rh.time.dt.month == month, drop=True)
                rh_parts.append(selected.groupby('time.year').mean(
                    'time', skipna=False).expand_dims(month=[month]))
            rh_monthly = xr.concat(rh_parts, dim='month', join='exact').load()
            humidity_status = 'available; missing or invalid daily values propagate to monthly NaN'
        except Exception as exc:
            humidity_status = f'humidity unavailable: {type(exc).__name__}: {exc}'
            print(humidity_status)
    variables['monthly_mean_relative_humidity'] = rh_monthly
    result = xr.Dataset(variables)
    for variable in labels:
        result[variable].attrs.update(units='days', frequency='monthly', temperature_statistic='daily mean temperature calculated as (tasmin + tasmax) / 2')
    result['monthly_precipitation'].attrs.update(units='inch', long_name='monthly total precipitation', temporal_aggregation='sum of daily precipitation')
    result['monthly_mean_relative_humidity'].attrs.update(units='percent', long_name='monthly mean relative humidity', daily_calculation='(rel_humid_min + rel_humid_max) / 2', temporal_aggregation='mean of daily mean relative humidity')
    result.attrs.update(reference_temperature_bin='55 to less than 65 degrees Fahrenheit', relative_humidity_file_directories='hursmin and hursmax', relative_humidity_internal_variables='rel_humid_min and rel_humid_max', relative_humidity_statistic='monthly mean of daily average relative humidity')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    result.attrs['humidity_status'] = humidity_status
    result.attrs['humidity_missing_policy'] = 'NaN; no imputation; temperature and precipitation retained'
    return result

## 5. Generate missing outputs

Existing runs are skipped. Humidity loading or calculation errors are printed and recorded in output metadata; humidity becomes NaN while other indicators are saved. Missing mandatory temperature or precipitation inputs still fail visibly. No fixed ensemble count is assumed.

New files are computed to a temporary file in the same output directory, then published using a no-overwrite hard link. This also protects an output that appears during processing. Temporary files are removed after each run.

In [ ]:
completed = skipped = failed = 0
for model, member, scenario in tqdm(jobs, desc=PAPER_NAME, unit='run'):
    if already_exists(model, member, scenario):
        skipped += 1
        continue
    inputs = {}
    result = None
    temporary = None
    try:
        inputs = common.load_paper_inputs(model, member, scenario, REQUIRED_VARIABLES)
        humidity_messages = []
        for variable in OPTIONAL_VARIABLES:
            try:
                loaded = common.load_paper_inputs(model, member, scenario, (variable,))
                inputs[variable] = loaded[variable]
            except Exception as exc:
                message = f'{variable}: {type(exc).__name__}: {exc}'
                humidity_messages.append(message)
                print(f'HUMIDITY NaN {model}/{member}/{scenario}: {message}')
        inputs['_humidity_status'] = '; '.join(humidity_messages) or 'loaded'
        annual = calculate_manderson_considine(inputs)
        result = common.pack_into_periods(annual)
        result.attrs.update(
            paper=PAPER_NAME, model=model, member=member, scenario=scenario,
            spatial_resolution='Illinois county',
            spatial_weighting='cosine-latitude area correction only',
            required_inputs=', '.join(REQUIRED_VARIABLES),
            optional_inputs=', '.join(OPTIONAL_VARIABLES),
        )
        target = output_path(model, member, scenario)
        with tempfile.NamedTemporaryFile(dir=output_directory, suffix='.partial', delete=False) as handle:
            temporary = Path(handle.name)
        result.to_netcdf(temporary, encoding={
            name: {'zlib': True, 'complevel': 4, 'shuffle': True}
            for name in result.data_vars
        })
        try:
            os.link(temporary, target)  # Fails rather than replacing an existing file.
        except FileExistsError:
            skipped += 1
            print('SKIPPED existing:', target.name)
        else:
            completed += 1
            existing_runs.add((model, member, scenario))
            print('WROTE:', target)
            print('Humidity:', result.attrs.get('humidity_status'))
    except Exception as exc:
        failed += 1
        print(f'FAILED {model}/{member}/{scenario}: {type(exc).__name__}: {exc}')
    finally:
        if result is not None:
            result.close()
        for data in inputs.values():
            if isinstance(data, xr.DataArray):
                data.close()
        if temporary is not None and temporary.exists():
            temporary.unlink()
        gc.collect()
print(f'New files={completed}; existing skipped={skipped}; failed={failed}')

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import xarray as xr

ROOT = Path(
    "/data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois"
)

MANDERSON_FOLDER = ROOT / "Manderson & Considine (2025)"
REFERENCE_FOLDER = ROOT / "Park (2016)"

# Required inputs for the humidity-inclusive Manderson workflow.
AUDIT_VARIABLES = ("tasmin", "tasmax", "pr", "hursmin", "hursmax")
SCENARIOS = ("ssp245", "ssp370")

EXCLUDE_MARKERS = (
    ".monthly.",
    ".yearly.",
    ".annual.",
    ".seasonal.",
    ".us_county_avg.",
)


def daily_files(folder):
    return [
        path
        for path in folder.rglob("*.nc")
        if not any(
            marker in path.name.lower()
            for marker in EXCLUDE_MARKERS
        )
    ] if folder.is_dir() else []


def output_inventory(folder):
    """Read run identifiers from existing output metadata."""
    runs = {}
    errors = []

    for path in sorted(folder.glob("*.nc")):
        try:
            with xr.open_dataset(
                path,
                decode_times=False,
                decode_timedelta=False,
            ) as ds:
                identifiers = tuple(
                    str(ds.attrs.get(key, "")).strip()
                    for key in ("model", "member", "scenario")
                )

                if not all(identifiers):
                    raise ValueError(
                        "Missing model/member/scenario attributes"
                    )

                runs.setdefault(identifiers, []).append({
                    "filename": path.name,
                    "has_humidity_output":
                        "monthly_mean_relative_humidity" in ds.data_vars,
                })

        except Exception as exc:
            errors.append({
                "filename": path.name,
                "error": f"{type(exc).__name__}: {exc}",
            })

    return runs, errors


def discover_input_runs(variable):
    """Discover future runs in variable/scenario/model/member folders."""
    runs = set()

    for scenario in SCENARIOS:
        folder = ROOT / variable / scenario

        if not folder.is_dir():
            continue

        for model_folder in folder.iterdir():
            if not model_folder.is_dir():
                continue

            for member_folder in model_folder.iterdir():
                if member_folder.is_dir() and daily_files(member_folder):
                    runs.add((
                        model_folder.name,
                        member_folder.name,
                        scenario,
                    ))

    return runs


manderson, manderson_errors = output_inventory(MANDERSON_FOLDER)
reference, reference_errors = output_inventory(REFERENCE_FOLDER)

# Union includes reference runs, delivered Manderson runs,
# and runs found in the daily input directory.
candidate_runs = set(manderson) | set(reference)

for variable in AUDIT_VARIABLES:
    candidate_runs |= discover_input_runs(variable)

rows = []

for model, member, scenario in sorted(candidate_runs):
    row = {
        "model": model,
        "member": member,
        "scenario": scenario,
        "manderson_output": (model, member, scenario) in manderson,
        "park_output": (model, member, scenario) in reference,
    }

    missing_inputs = []

    for variable in AUDIT_VARIABLES:
        for experiment in ("historical", scenario):
            folder = ROOT / variable / experiment / model / member
            available = bool(daily_files(folder))

            label = (
                f"{variable}_historical"
                if experiment == "historical"
                else f"{variable}_future"
            )

            row[label] = available

            if not available:
                missing_inputs.append(f"{variable}/{experiment}")

    humidity_missing = [
        item for item in missing_inputs
        if item.startswith(("hursmin/", "hursmax/"))
    ]
    other_missing = [
        item for item in missing_inputs
        if not item.startswith(("hursmin/", "hursmax/"))
    ]

    row["missing_inputs"] = ", ".join(missing_inputs)
    row["all_input_files_present"] = not missing_inputs

    if row["manderson_output"]:
        row["status"] = "Manderson output exists"
    elif humidity_missing and not other_missing:
        row["status"] = "Missing humidity inputs only"
    elif missing_inputs:
        row["status"] = "Missing other or multiple inputs"
    else:
        row["status"] = "All input files present; Manderson output absent"

    rows.append(row)

audit = pd.DataFrame(rows)

print("MANDERSON OUTPUT INVENTORY")
print("Files:", sum(len(paths) for paths in manderson.values()))
print("Unique model-member-scenario runs:", len(manderson))
print("Unique models:", len({run[0] for run in manderson}))
print("Unique model-member pairs:", len({run[:2] for run in manderson}))
print("Runs by scenario:", dict(Counter(run[2] for run in manderson)))

print("\nPARK REFERENCE INVENTORY")
print("Files:", sum(len(paths) for paths in reference.values()))
print("Unique model-member-scenario runs:", len(reference))

print("\nINPUT DIRECTORY CHECK")
for variable in AUDIT_VARIABLES:
    print(f"{variable}: {(ROOT / variable).exists()}")

if not audit.empty:
    comparison = audit.loc[
        audit["park_output"] & ~audit["manderson_output"]
    ].copy()

    print("\nPARK RUNS WITHOUT MANDERSON OUTPUTS:", len(comparison))

    if not comparison.empty:
        display(
            comparison.groupby(["scenario", "status"])
            .size()
            .rename("runs")
            .reset_index()
        )

        display(
            comparison[
                ["model", "member", "scenario", "status", "missing_inputs"]
            ].reset_index(drop=True)
        )

    unexplained = audit.loc[
        ~audit["manderson_output"]
        & audit["all_input_files_present"]
    ]

    print(
        "\nRUNS WITH ALL REQUIRED INPUT FILES BUT NO MANDERSON OUTPUT:",
        len(unexplained),
    )

    if not unexplained.empty:
        display(
            unexplained[
                ["model", "member", "scenario", "park_output"]
            ].reset_index(drop=True)
        )

print("\nOUTPUT READ ERRORS")
if manderson_errors or reference_errors:
    display(pd.DataFrame(
        [{"paper": "Manderson", **item} for item in manderson_errors]
        + [{"paper": "Park", **item} for item in reference_errors]
    ))
else:
    print("None")

duplicates = [
    {
        "paper": paper,
        "model": run[0],
        "member": run[1],
        "scenario": run[2],
        "file_count": len(paths),
    }
    for paper, inventory in (
        ("Manderson", manderson),
        ("Park", reference),
    )
    for run, paths in inventory.items()
    if len(paths) > 1
]

if duplicates:
    print("\nDUPLICATE OUTPUT RUNS")
    display(pd.DataFrame(duplicates))

print(
    "\nThis checks file presence, not complete date coverage or input quality. "
    "Runs with all inputs present but no output need processing-log review. "
    "Missing inputs describe the current directory contents."
)

In [ ]:
import re
import time
from urllib.parse import urljoin, urlparse, unquote

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

LOCA2_SOURCE = "https://cirrus.ucsd.edu/~pierce/LOCA2/NAmer/"
GRID = "0p0625deg"
HUMIDITY_VARIABLES = ("hursmin", "hursmax")

if "audit" not in globals():
    raise NameError("Run the previous local-input audit cell first.")

# Check humidity for reference runs absent from Manderson.
missing_runs = audit.loc[
    audit["park_output"] & ~audit["manderson_output"],
    ["model", "member", "scenario"],
].drop_duplicates()

session = requests.Session()
cache = {}


def inspect_directory(url):
    """Read one archive listing; do not download NetCDF files."""
    if url in cache:
        return cache[url]

    try:
        response = session.get(url, timeout=(20, 90))

        if response.status_code == 404:
            result = {
                "status": "HTTP 404",
                "files": [],
                "detail": "Directory not found at this URL",
            }

        elif response.status_code != 200:
            result = {
                "status": "UNRESOLVED",
                "files": [],
                "detail": f"HTTP {response.status_code}",
            }

        else:
            soup = BeautifulSoup(response.text, "html.parser")
            files = set()

            for anchor in soup.find_all("a", href=True):
                link = urljoin(response.url, anchor["href"])
                filename = unquote(
                    urlparse(link).path.rsplit("/", 1)[-1]
                )
                lower = filename.lower()

                if not lower.endswith(".nc"):
                    continue

                if any(
                    marker in lower
                    for marker in (
                        ".monthly.",
                        ".yearly.",
                        ".annual.",
                        ".seasonal.",
                        ".us_county_avg.",
                    )
                ):
                    continue

                files.add(filename)

            # Avoid treating a generic HTML response as an empty listing.
            listing_text = soup.get_text(" ", strip=True).lower()
            recognizable_listing = (
                "index of" in listing_text
                or "parent directory" in listing_text
            )

            if files:
                status = "DAILY FILES LISTED"
                detail = ""
            elif recognizable_listing:
                status = "NO DAILY FILES LISTED"
                detail = "Directory exists but has no matching daily NetCDF links"
            else:
                status = "UNRESOLVED"
                detail = "HTTP 200, but response is not a recognizable directory listing"

            result = {
                "status": status,
                "files": sorted(files),
                "detail": detail,
            }

    except requests.RequestException as exc:
        result = {
            "status": "UNRESOLVED",
            "files": [],
            "detail": f"{type(exc).__name__}: {exc}",
        }

    cache[url] = result
    time.sleep(0.2)
    return result


records = []

for row in missing_runs.itertuples(index=False):
    for experiment in ("historical", row.scenario):
        for variable in HUMIDITY_VARIABLES:
            url = (
                f"{LOCA2_SOURCE}{row.model}/{GRID}/"
                f"{row.member}/{experiment}/{variable}/"
            )

            result = inspect_directory(url)

            # Filename year ranges are useful for screening only.
            ranges = sorted({
                match.group(0)
                for filename in result["files"]
                for match in re.finditer(
                    r"(?<!\d)\d{4}-\d{4}(?!\d)", filename
                )
            })

            records.append({
                "model": row.model,
                "member": row.member,
                "scenario": row.scenario,
                "experiment": experiment,
                "variable": variable,
                "source_status": result["status"],
                "daily_files_listed": len(result["files"]),
                "filename_year_ranges": ", ".join(ranges),
                "filenames": "; ".join(result["files"]),
                "url": url,
                "detail": result["detail"],
            })

    print(f"Checked: {row.model}/{row.member}/{row.scenario}")

source_audit = pd.DataFrame(records)

if source_audit.empty:
    print("No Park runs are missing from the Manderson output inventory.")
else:
    print("\nSOURCE CHECK SUMMARY")
    display(
        source_audit.groupby(
            ["experiment", "variable", "source_status"]
        )
        .size()
        .rename("checks")
        .reset_index()
    )

    print("\nDETAILED RESULTS")
    display(source_audit[
        [
            "model", "member", "scenario", "experiment", "variable",
            "source_status", "daily_files_listed", "filename_year_ranges",
        ]
    ])

    print("\nURLS REQUIRING REVIEW")
    display(
        source_audit.loc[
            source_audit["source_status"] != "DAILY FILES LISTED",
            ["model", "member", "experiment", "variable",
             "source_status", "url", "detail"],
        ].drop_duplicates().reset_index(drop=True)
    )